# 05 · Medallion
*Three vaults: raw as received (bronze), cleaned (silver), briefings for the Avengers (gold).*

Build silver and gold from the bronze tables of chapter 03. This is a simple full-refresh version;
chapters 06 and 08 make it safer and incremental.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

## Silver: typed, deduplicated incidents (no quality gate yet)

In [ ]:
bronze = spark.table("shield.bronze.incident_reports")

latest = Window.partitionBy("incident_id").orderBy(F.col("_ingested_at").desc())
silver = (
    bronze.withColumn("reported_ts", F.expr("try_to_timestamp(reported_at, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")"))
    .withColumn("rn", F.row_number().over(latest))
    .filter("rn = 1")
    .drop("rn")
)
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("shield.silver.incident_reports")
print(spark.table("shield.silver.incident_reports").count())

## Silver: locations

In [ ]:
(
    spark.table("shield.bronze.locations")
    .select("location_id", "city", "country", "lat", "lon")
    .write.mode("overwrite").saveAsTable("shield.silver.locations")
)

## Gold: business-ready aggregates

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.gold.daily_threat_briefing AS
    SELECT
      CAST(i.reported_ts AS DATE) AS report_date,
      l.city,
      l.country,
      i.incident_type,
      i.severity,
      COUNT(*)            AS incidents,
      SUM(i.casualties)   AS casualties
    FROM shield.silver.incident_reports i
    JOIN shield.silver.locations l USING (location_id)
    WHERE i.reported_ts IS NOT NULL
    GROUP BY ALL
    """
)
display(spark.sql("SELECT * FROM shield.gold.daily_threat_briefing ORDER BY report_date DESC LIMIT 10"))

## Gold: location risk ranking

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.gold.location_risk AS
    SELECT
      l.city,
      COUNT(*) AS incidents,
      SUM(CASE WHEN i.severity IN ('high','critical') THEN 1 ELSE 0 END) AS serious_incidents,
      ROUND(AVG(i.casualties), 2) AS avg_casualties
    FROM shield.silver.incident_reports i
    JOIN shield.silver.locations l USING (location_id)
    GROUP BY l.city
    """
)
display(spark.table("shield.gold.location_risk").orderBy(F.desc("serious_incidents")))

Notice what is **wrong** with this silver table: invalid severities, future timestamps and negative casualties
flowed straight through, and they are now in the gold numbers. That is chapter 06.

In [ ]:
display(spark.sql("SELECT severity, COUNT(*) FROM shield.silver.incident_reports GROUP BY severity"))

## Try it yourself
1. Add `gold.agent_workload`: incidents per agent codename (join `silver.agents`).
2. Which of the silver transformations is not idempotent if you run the notebook twice? (Hint: it is, why?)
3. Sketch what each layer would hold for a retail dataset you know.